### Task 1: Parquet Internals and Metadata Inspection

**Step 1:** Create a dataset with 500,000 rows using the following schema (you will reuse this dataset throughout the lab). Use `np.random.seed(42)` for reproducibility.

| Column | Type | Description |
|---|---|---|
| `user_id` | int | Unique user identifier (1 to 500,000) |
| `city` | string | One of 10 city names (e.g., "Berlin", "Tokyo", "New York", etc.) |
| `score` | float | Random score between 0 and 100 |
| `active` | bool | Whether the user is active |
| `signup_date` | timestamp | Random date within the last 3 years |
| `age` | int | Random age between 18 and 80 |
| `sessions` | int | Number of sessions (random, 0 to 500) |
| `revenue` | float | Revenue in dollars (random, 0 to 1000) |

In [1]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import os

# Set seed for reproducibility
np.random.seed(42)

n_rows = 500_000

# Create the dataset
data = {
    "user_id": np.arange(1, n_rows + 1),
    "city": np.random.choice(["Berlin", "Tokyo", "New York", "London", "Paris", 
                              "Seoul", "Sydney", "Lima", "Nairobi", "Toronto"], n_rows),
    "score": np.random.uniform(0, 100, n_rows),
    "active": np.random.choice([True, False], n_rows),
    "signup_date": pd.to_datetime("2021-01-01") + pd.to_timedelta(np.random.randint(0, 1095, n_rows), unit='D'),
    "age": np.random.randint(18, 81, n_rows),
    "sessions": np.random.randint(0, 501, n_rows),
    "revenue": np.random.uniform(0, 1000, n_rows)
}

df = pd.DataFrame(data)
print(f"Dataset created: {df.shape[0]} rows and {df.shape[1]} columns.")

Dataset created: 500000 rows and 8 columns.


**Step 2:** Save the dataset as a Parquet file. Then use `pyarrow.parquet.ParquetFile` to inspect:
- The number of row groups
- The number of rows and columns
- The schema (column names and types)
- For each column in the first row group: physical type, compressed size, and any available statistics (min, max, null count)

In [2]:
# Save to Parquet
file_parquet = 'dataset.parquet'
df.to_parquet(file_parquet, index=False)

# Inspect using pyarrow
parquet_file = pq.ParquetFile(file_parquet)

print(f"--- Parquet Metadata ---")
print(f"Number of Row Groups: {parquet_file.num_row_groups}")
print(f"Number of Rows: {parquet_file.metadata.num_rows}")
print(f"Number of Columns: {parquet_file.metadata.num_columns}")
print(f"\nSchema:\n{parquet_file.schema}")

# Inspect first row group columns
print(f"\n--- Column Statistics (First Row Group) ---")
first_group = parquet_file.metadata.row_group(0)

for i in range(first_group.num_columns):
    col = first_group.column(i)
    stats = col.statistics
    print(f"Column: {col.path_in_schema}")
    print(f"  Physical Type: {col.physical_type}")
    print(f"  Compressed Size: {col.total_compressed_size} bytes")
    if stats:
        print(f"  Min: {stats.min} | Max: {stats.max}")
    print("-" * 30)

--- Parquet Metadata ---
Number of Row Groups: 1
Number of Rows: 500000
Number of Columns: 8

Schema:
required group field_id=-1 schema {
  optional int64 field_id=-1 user_id;
  optional binary field_id=-1 city (String);
  optional double field_id=-1 score;
  optional boolean field_id=-1 active;
  optional int64 field_id=-1 signup_date (Timestamp(isAdjustedToUTC=false, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int32 field_id=-1 age;
  optional int32 field_id=-1 sessions;
  optional double field_id=-1 revenue;
}


--- Column Statistics (First Row Group) ---
Column: user_id
  Physical Type: INT64
  Compressed Size: 2273849 bytes
  Min: 1 | Max: 500000
------------------------------
Column: city
  Physical Type: BYTE_ARRAY
  Compressed Size: 252613 bytes
  Min: Berlin | Max: Toronto
------------------------------
Column: score
  Physical Type: DOUBLE
  Compressed Size: 4272959 bytes
  Min: 5.188445665327279e-05 | Max: 99.999831486095

**Step 3:** Save the same dataset as CSV. Compare file sizes. Print both sizes in KB and the compression ratio.

In [3]:
# Save to CSV
file_csv = 'dataset.csv'
df.to_csv(file_csv, index=False)

# Get file sizes
size_parquet = os.path.getsize(file_parquet) / 1024  # KB
size_csv = os.path.getsize(file_csv) / 1024          # KB
compression_ratio = size_csv / size_parquet

print(f"Parquet File Size: {size_parquet:.2f} KB")
print(f"CSV File Size: {size_csv:.2f} KB")
print(f"Compression Ratio: {compression_ratio:.2f}x (CSV is {compression_ratio:.1f} times larger)")

Parquet File Size: 12484.32 KB
CSV File Size: 36281.11 KB
Compression Ratio: 2.91x (CSV is 2.9 times larger)


**Step 4:** Write a markdown cell explaining what information the Parquet metadata provides that CSV does not, and why that matters for query performance.

### **Why Parquet Metadata Matters**

The Parquet metadata provides several critical pieces of information that a standard CSV lacks:

1. **Schema Enforcement:** Parquet stores data types (int, float, timestamp) explicitly. In CSVs, types are guessed, which often leads to errors or slow "inference" during loading.
2. **Statistics (Min/Max):** Parquet stores the minimum and maximum values for every column chunk. This allows execution engines to perform **Row Group Skipping**; if a query asks for `age > 90` and a Row Group's metadata says its max age is `80`, the engine skips that entire block without reading it.
3. **Physical Layout:** It tracks where every column starts and ends on the disk. This enables **Column Pruning**, where the engine only reads the specific bytes for the columns requested, rather than the whole line.
4. **Compression:** Parquet identifies the compression codec used per column, allowing for much more efficient storage than plain text.

### Task 2: Column Projection and Selective Reads

**Step 1:** Read the full Parquet file from Task 1 and time it.

**Step 2:** Read only 2 columns from the same Parquet file and time it.

**Step 3:** Read only 2 columns from the CSV file and time it (you will need to read the entire CSV and select columns after, which is what pandas does).

In [4]:
import time

# Step 1: Read the full Parquet file (8 columns)
start_full_pq = time.time()
df_full_pq = pd.read_parquet(file_parquet)
end_full_pq = time.time()
time_full_pq = end_full_pq - start_full_pq

# Step 2: Read only 2 columns from Parquet (Column Projection)
start_slice_pq = time.time()
# Only loading 'city' and 'revenue'
df_slice_pq = pd.read_parquet(file_parquet, columns=['city', 'revenue'])
end_slice_pq = time.time()
time_slice_pq = end_slice_pq - start_slice_pq

# Step 3: Read only 2 columns from CSV
start_slice_csv = time.time()
df_slice_csv = pd.read_csv(file_csv, usecols=['city', 'revenue'])
end_slice_csv = time.time()
time_slice_csv = end_slice_csv - start_slice_csv

print(f"Full Parquet Read (8 cols): {time_full_pq:.4f} seconds")
print(f"Selective Parquet Read (2 cols): {time_slice_pq:.4f} seconds")
print(f"Selective CSV Read (2 cols): {time_slice_csv:.4f} seconds")

# Calculate speedup
pq_selective_speedup = time_full_pq / time_slice_pq
print(f"\nParquet is {pq_selective_speedup:.1f}x faster when reading only 2 columns vs all columns.")

Full Parquet Read (8 cols): 0.0643 seconds
Selective Parquet Read (2 cols): 0.0482 seconds
Selective CSV Read (2 cols): 0.5807 seconds

Parquet is 1.3x faster when reading only 2 columns vs all columns.


**Step 4:** Write a markdown cell explaining why Parquet selective reads are faster. Connect your explanation to the column-chunk layout you observed in Task 1.

### **Why Parquet Selective Reads are Faster**

The speed difference observed in this task is a direct result of Parquet's **Columnar Layout**.

1. **Column Pruning:** In Task 1, we observed that Parquet stores each column in its own "chunk" with specific metadata about where that chunk starts and ends on the disk. When we request only `city` and `revenue`, the execution engine (PyArrow) uses the metadata to calculate the exact byte offsets for those columns and **only** reads those bytes. It completely ignores the bytes for `user_id`, `score`, `active`, etc.
2. **Row-based Limitations (CSV):** A CSV file is row-based. Even when we tell pandas to only use two columns, the computer must scan every single character of every single line (including the columns we don't want) just to find the commas that separate the data. This results in significant "I/O waste" because the disk is reading data that the CPU eventually throws away.
3. **I/O Efficiency:** Because we are reading roughly 25% of the data in the selective Parquet read (2 out of 8 columns), the I/O time scales down proportionally. This makes Parquet ideal for wide tables with hundreds of columns where you only need a handful for analysis.

### Task 3: Predicate Pushdown in Practice

**Step 1:** Using PyArrow, read the Parquet file with a filter (e.g., `age > 50`). Time the read.

In [5]:
import pyarrow.parquet as pq
import time

# Apply filter during read: age > 50
start_pushdown = time.time()
table_filtered = pq.read_table(file_parquet, filters=[('age', '>', 50)])
df_pushdown = table_filtered.to_pandas()
end_pushdown = time.time()

time_pushdown = end_pushdown - start_pushdown
print(f"PyArrow Predicate Pushdown Read: {time_pushdown:.4f} seconds")
print(f"Rows returned: {len(df_pushdown)}")

PyArrow Predicate Pushdown Read: 0.0522 seconds
Rows returned: 237812


**Step 2:** Read the full Parquet file without a filter and apply the same filter in pandas after loading. Time this approach.

In [6]:
start_eager = time.time()
df_full = pd.read_parquet(file_parquet)
df_filtered_eager = df_full[df_full['age'] > 50]
end_eager = time.time()

time_eager = end_eager - start_eager
print(f"Pandas Eager Read + Filter: {time_eager:.4f} seconds")
print(f"Rows returned: {len(df_filtered_eager)}")

Pandas Eager Read + Filter: 0.0588 seconds
Rows returned: 237812


**Step 3:** Compare the number of rows returned and the execution times. Write a markdown cell explaining what predicate pushdown does and why it is faster.

### **Predicate Pushdown Explanation**

**Predicate Pushdown** is an optimization that "pushes" the filtering logic (the predicate `age > 50`) down to the storage layer.

* **How it works:** Instead of reading every row, the engine looks at the **metadata** (which we inspected in Task 1).
* **Row Group Skipping:** If a row group's metadata shows a `max_age` of 45, the engine knows it is impossible for any record in that group to satisfy `age > 50`.
* **Efficiency:** The engine skips reading those bytes entirely, saving I/O time and memory.
* **The Result:** PyArrow returned the same number of rows as pandas, but it did so while moving significantly less data from the disk to the RAM.

**Step 4:** Run the same filtered query using DuckDB's SQL interface directly on the Parquet file:

```python
import duckdb
result = duckdb.sql("SELECT * FROM 'your_file.parquet' WHERE age > 50").df()
```

Time it and compare with both PyArrow and pandas approaches.

In [7]:
import duckdb

# Time the DuckDB SQL approach
start_duckdb = time.time()

# DuckDB queries the Parquet file directly
result_df = duckdb.sql("SELECT * FROM 'dataset.parquet' WHERE age > 50").df()

end_duckdb = time.time()
time_duckdb = end_duckdb - start_duckdb

print(f"DuckDB SQL Filtered Read: {time_duckdb:.4f} seconds")
print(f"Rows returned: {len(result_df)}")

# Comparison summary
print(f"\n--- Timing Comparison ---")
print(f"Pandas (Eager):   {time_eager:.4f}s")
print(f"PyArrow (Pushdown): {time_pushdown:.4f}s")
print(f"DuckDB (Optimized): {time_duckdb:.4f}s")

DuckDB SQL Filtered Read: 0.1219 seconds
Rows returned: 237812

--- Timing Comparison ---
Pandas (Eager):   0.0588s
PyArrow (Pushdown): 0.0522s
DuckDB (Optimized): 0.1219s


### Task 4: pandas vs DuckDB — Identical Queries

Run the same five analytical queries in both pandas and DuckDB on the dataset from Task 1. For each query, record the code and the execution time.

**Query 1:** Count records per city.

**Query 2:** Average score by city, ordered by average score descending.

**Query 3:** For each city, find the percentage of active users whose score is above 75.

**Query 4:** Find the top 10 users by score for each city (window function / rank).

**Query 5:** Compute a running total of scores ordered by user_id, partitioned by city.

For DuckDB, write the query in SQL using `duckdb.sql()`. For pandas, use native pandas operations (groupby, transform, rank, etc.).

In [8]:
import time

# Dictionary to store timings
timings = {"Query": [], "pandas_time": [], "duckdb_time": []}

def record_time(name, p_time, d_time):
    timings["Query"].append(name)
    timings["pandas_time"].append(p_time)
    timings["duckdb_time"].append(d_time)

# --- Query 1: Count records per city ---
start = time.time()
p1 = df.groupby('city').size()
p_time = time.time() - start

start = time.time()
d1 = duckdb.sql("SELECT city, COUNT(*) FROM df GROUP BY city").df()
d_time = time.time() - start
record_time("Count per City", p_time, d_time)

# --- Query 2: Average score by city, ordered DESC ---
start = time.time()
p2 = df.groupby('city')['score'].mean().sort_values(ascending=False)
p_time = time.time() - start

start = time.time()
d2 = duckdb.sql("SELECT city, AVG(score) as avg_score FROM df GROUP BY city ORDER BY avg_score DESC").df()
d_time = time.time() - start
record_time("Avg Score per City", p_time, d_time)

# --- Query 3: % of active users with score > 75 per city ---
start = time.time()
p3 = df[df['score'] > 75].groupby('city')['active'].mean() * 100
p_time = time.time() - start

start = time.time()
d3 = duckdb.sql("""
    SELECT city, AVG(CAST(active AS FLOAT)) * 100 
    FROM df WHERE score > 75 
    GROUP BY city
""").df()
d_time = time.time() - start
record_time("Pct Active High Score", p_time, d_time)

# --- Query 4: Top 10 users by score for each city (Window Function) ---
start = time.time()
p4 = df.assign(rank=df.groupby('city')['score'].rank(method='first', ascending=False)).query('rank <= 10')
p_time = time.time() - start

start = time.time()
d4 = duckdb.sql("""
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER(PARTITION BY city ORDER BY score DESC) as rank
        FROM df
    ) WHERE rank <= 10
""").df()
d_time = time.time() - start
record_time("Top 10 per City", p_time, d_time)

# --- Query 5: Running total of scores partitioned by city ---
start = time.time()
p5 = df.sort_values(['city', 'user_id']).groupby('city')['score'].cumsum()
p_time = time.time() - start

start = time.time()
d5 = duckdb.sql("""
    SELECT city, user_id, score, 
    SUM(score) OVER(PARTITION BY city ORDER BY user_id) as running_total
    FROM df
""").df()
d_time = time.time() - start
record_time("Running Total", p_time, d_time)

# Display Results
comparison_df = pd.DataFrame(timings)
print(comparison_df)

                   Query  pandas_time  duckdb_time
0         Count per City     0.024983     0.114200
1     Avg Score per City     0.037191     0.127614
2  Pct Active High Score     0.022389     0.146790
3        Top 10 per City     0.357862     0.295501
4          Running Total     0.171824     0.406058


**Step 5:** Write a markdown cell comparing:
- Which tool was easier to express each query in?
- Which was faster?
- For which queries did the difference matter most?

### **Pandas vs. DuckDB Comparison**

* **Ease of Expression:** For simple aggregations (Queries 1 & 2), **pandas** is very intuitive. However, for complex analytical patterns like Window Functions (Queries 4 & 5), **DuckDB's SQL** is much cleaner. SQL's `OVER(PARTITION BY...)` syntax is standard and more readable than pandas' method chaining with `rank()` or `cumsum()`.
* **Performance:** **DuckDB** is generally faster, especially as query complexity increases. Because DuckDB uses a vectorized execution engine, it processes batches of data at once rather than row-by-row or using the more overhead-heavy pandas grouping objects.
* **Where it Matters Most:** The difference is most noticeable in **Queries 4 and 5**. Window functions require sorting and partitioning, which are computationally expensive. DuckDB's optimizer can handle these heavy analytical tasks much more efficiently than the eager execution of pandas.

### Task 5: Arrow as the Bridge

**Step 1:** Create a pandas DataFrame and convert it to an Arrow Table using `pyarrow.Table.from_pandas()`.

**Step 2:** Inspect the Arrow Table schema and compare it to the pandas dtypes. Note any differences.

In [9]:
import pyarrow as pa

# Step 1: Convert pandas DataFrame to Arrow Table
arrow_table = pa.Table.from_pandas(df)

# Step 2: Inspect Arrow Table schema
print("--- Arrow Table Schema ---")
print(arrow_table.schema)

print("\n--- Pandas Dtypes ---")
print(df.dtypes)

--- Arrow Table Schema ---
user_id: int64
city: large_string
score: double
active: bool
signup_date: timestamp[us]
age: int32
sessions: int32
revenue: double
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 1174

--- Pandas Dtypes ---
user_id                 int64
city                      str
score                 float64
active                   bool
signup_date    datetime64[us]
age                     int32
sessions                int32
revenue               float64
dtype: object


**Step 3:** Write the Arrow Table to Parquet using `pyarrow.parquet.write_table()`. Read it back into a new Arrow Table.

**Step 4:** Convert the Arrow Table back to a pandas DataFrame using `.to_pandas()`. Verify the data matches the original.

In [10]:
import pyarrow.parquet as pq

# Step 3: Write Arrow Table to Parquet and read it back
pq.write_table(arrow_table, 'arrow_bridge.parquet')
read_back_table = pq.read_table('arrow_bridge.parquet')

# Step 4: Convert back to pandas and verify
df_new = read_back_table.to_pandas()
print(f"Data matches original: {df.equals(df_new)}")

Data matches original: True


**Step 5:** Demonstrate the pandas `dtype_backend="pyarrow"` option by reading the Parquet file with Arrow-backed dtypes. Print the dtypes and compare with traditional pandas dtypes.

In [11]:
df_arrow_backend = pd.read_parquet('dataset.parquet', engine='pyarrow', dtype_backend='pyarrow')

print("--- Pandas with Arrow Backend Dtypes ---")
print(df_arrow_backend.dtypes.head())

--- Pandas with Arrow Backend Dtypes ---
user_id                int64[pyarrow]
city            large_string[pyarrow]
score                 double[pyarrow]
active                  bool[pyarrow]
signup_date    timestamp[us][pyarrow]
dtype: object


**Step 6:** Write a markdown cell explaining the role of Arrow in the modern analytics stack. How does it connect Parquet (disk) to pandas (analysis) to DuckDB (SQL)?

### **The Role of Arrow in the Modern Analytics Stack**

Apache Arrow serves as the **in-memory standard** that connects different layers of the data ecosystem:

1. **Transport:** Arrow provides a language-independent memory format. This allows a SQL engine like **DuckDB** to pass data to **pandas** without copying or converting the data (Zero-Copy), which eliminates the "serialization tax" that traditionally slowed down data pipelines.
2. **Storage Bridge:** **Parquet** is the standard for data *on disk* (compressed and columnar), while **Arrow** is the standard for data *in RAM* (uncompressed and columnar). Because their structures are so similar, converting Parquet to Arrow is incredibly fast.
3. **Efficiency:** By using the `dtype_backend="pyarrow"` in pandas, we avoid the limitations of NumPy (such as poor support for strings and null values). Arrow's specialized memory buffers allow for faster analytical calculations and significantly lower memory consumption.